***Dependencies***

---



In [ ]:
!pip install -q ultralytics pyyaml

***Code***

---



In [16]:
!pip install -q ultralytics pyyaml

import os
import glob
import zipfile
import shutil
import yaml
import urllib.request
from ultralytics import YOLO

# 1. Paths Setup
dataset_dir = "/content/dataset"
target_zip = "/content/TheDataset.zip"
yaml_path = os.path.join(dataset_dir, "data.yaml")
train_txt = os.path.join(dataset_dir, "train.txt")
weights_path = "/content/yolov8s.pt"

# 2. Extract & Reorganize Dataset if needed
if not os.path.exists(yaml_path):
    found_zips = glob.glob("/content/*.zip")
    other_zips = [f for f in found_zips if os.path.basename(f) != "TheDataset.zip"]
    if other_zips:
        if os.path.exists(target_zip): os.remove(target_zip)
        os.rename(other_zips[0], target_zip)

    if os.path.exists("/content/data.yaml"):
        os.makedirs(dataset_dir, exist_ok=True)
        for item in ["data.yaml", "train.txt", "images", "labels"]:
            src, dst = os.path.join("/content", item), os.path.join(dataset_dir, item)
            if os.path.exists(src) and not os.path.exists(dst): shutil.move(src, dst)
    elif os.path.exists(target_zip):
        os.makedirs(dataset_dir, exist_ok=True)
        with zipfile.ZipFile(target_zip, 'r') as zip_ref:
            zip_ref.extractall(dataset_dir)

# 3. Fix train.txt and data.yaml
if os.path.exists(train_txt):
    with open(train_txt, "r") as f:
        lines = [line.strip() for line in f if line.strip()]
    fixed_lines = [os.path.join(dataset_dir, l.lstrip("./").replace("data/", "", 1)) + "\n" if not l.startswith("/") else l + "\n" for l in lines]
    with open(train_txt, "w") as f: f.writelines(fixed_lines)

with open(yaml_path, "r") as f: config = yaml.safe_load(f)
config["path"], config["train"], config["val"] = dataset_dir, "train.txt", "train.txt"
with open(yaml_path, "w") as f: yaml.dump(config, f)

# 4. Pre-download YOLOv8 Small Base Weights
if not os.path.exists(weights_path):
    print("⬇️ Pre-downloading yolov8s.pt...")
    url = "https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8s.pt"
    urllib.request.urlretrieve(url, weights_path)

# 5. Launch Upgraded Training
model = YOLO(weights_path)

results = model.train(
    data=yaml_path,
    epochs=100,          # Increased from 50
    imgsz=800,           # Higher resolution for small leaf lesions
    batch=16,            # Reduce to 8 if Colab throws an Out Of Memory (OOM) error
    patience=20,         # Early stopping if no improvement after 20 epochs
    degrees=15.0,        # Rotation augmentation
    fliplr=0.5,          # Horizontal flip augmentation
    mosaic=1.0,          # Mosaic composite training
    project="leaf_disease_run",
    name="train_v2",
    exist_ok=True
)

⬇️ Pre-downloading yolov8s.pt...
Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=15.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_v2, nbs

***Export***

---



In [17]:
import shutil
from google.colab import files

file = '_v2'

# 1. Zip the train-4 directory
shutil.make_archive(f'train-{file}', 'zip', f'/content/runs/detect/leaf_disease_run/train{file}')

# 2. Trigger browser download
files.download(f'train-{file}.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

***Delete Training Data***

---



In [ ]:
!rm -rf /content/runs/detect/leaf_disease_run/train

***Create train.py***

---



In [ ]:
import os
import shutil

# Create src directory
os.makedirs("src", exist_ok=True)

train_code = '''
import os
import yaml
import urllib.request
from ultralytics import YOLO

def train():
    dataset_dir = "/content/dataset"
    yaml_path = os.path.join(dataset_dir, "data.yaml")
    weights_path = "yolov8n.pt"

    # Pre-download base weights if missing
    if not os.path.exists(weights_path):
        print("⬇️ Downloading base model weights (yolov8n.pt)...")
        url = "https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8n.pt"
        urllib.request.urlretrieve(url, weights_path)

    # Initialize and train
    model = YOLO(weights_path)
    results = model.train(
        data=yaml_path,
        epochs=50,
        imgsz=640,
        batch=16,
        project="leaf_disease_run",
        name="train",
        exist_ok=True
    )
    print("✅ Training complete!")

if __name__ == "__main__":
    train()
'''

with open("src/train.py", "w") as f:
    f.write(train_code)

print("✅ Created src/train.py")

***Create predict.py***

---



In [ ]:
predict_code = '''
import sys
from ultralytics import YOLO

def predict(image_path, weights_path="best.pt"):
    # Load trained model
    model = YOLO(weights_path)

    # Perform inference
    results = model(image_path)

    # Display bounding box output
    results[0].show()

    # Save output image
    results[0].save(filename="prediction_output.jpg")
    print("✅ Prediction saved to prediction_output.jpg")

if __name__ == "__main__":
    img_path = sys.argv[1] if len(sys.argv) > 1 else "test.jpg"
    predict(img_path)
'''

with open("src/predict.py", "w") as f:
    f.write(predict_code)

print("✅ Created src/predict.py")